# ElasticSearchへの登録のシンプルなサンプル

ElasticSearchクライアントを使用するサンプル。

※ElasticSearchおよびクライアントのバージョン：8.18.0

ただし、今後、ElasticSearchの最新版を使う場合、ElasticSearchクライアントの対応まで時間がかかることがあり、また、互換性がなく修正が必要になる可能性がある。

## 必要パッケージのインポート

In [ ]:
from elasticsearch8 import Elasticsearch
from sentence_transformers import SentenceTransformer
import time

## 設定

In [ ]:
ES_URL = 'http://llm-rag-examples-elasticsearch1:9200'
INDEX_NAME = 'vector_test02'

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## ElasticSearchオブジェクト生成

In [ ]:
es = Elasticsearch(
    [ES_URL],
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
)

## インデックスの有無チェック、あれば削除

In [ ]:
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'index {INDEX_NAME} is deleted.')
else:
    print(f'index {INDEX_NAME} does not exist.')

In [ ]:
mapping = {
    'mappings': {
        'properties': {
            'text': {'type': 'text'},
            'vector': {
                'type': 'dense_vector',
                'dims': MODEL_DIM,    # モデルの次元数
                'index': True,
                'similarity': 'cosine'
            }
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)
print(f"Index '{INDEX_NAME}' created.")

## ドキュメントをインデックス

In [ ]:
# --- 登録するテキストデータ ---
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

In [ ]:
for i, text in enumerate(texts):
    vector = model.encode(text)
    doc = {
        'text': text,
        'vector': vector.tolist()
    }
    es.index(index=INDEX_NAME, id=i, document=doc)
    print(f"Indexed: {text}")